<a href="https://colab.research.google.com/github/joshtburdick/misc/blob/master/plog/Fermat_style_factoring_using_a_RNS%2C_take_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Questions about Fermat factoring using a RNS (take 4, or so)

This is a follow-on to my [previous attempt at factoring using loopy belief propagation](https://github.com/joshtburdick/misc/blob/master/plog/Factoring_using_loopy_belief_propagation.ipynb)

Here, we ask, for $x$ a product of two primes not in $p_i$:
- how many different solutions are there for $x = a^2 - b^2$?
- how often does $a+b|x$ ? (Presumably not often.)
- how often is $\mathrm{GCD}(a+b, x)$ nontrivial (that is, neither 1 or $x$) ?

In [88]:
import math

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sympy

from tqdm import tqdm

In [89]:
# courtesy Gemini
def find_relatively_prime(primes):
    """
    Takes a list of primes and returns the numbers in [1, product of primes-1]
    which are relatively prime to all of them.

    Args:
        primes (list): A list of prime numbers.

    Returns:
        list: A list of numbers from 1 to (product of primes - 1) that are
              relatively prime to all given primes.
    """
    if not primes:
        return []

    product_of_primes = 1
    for p in primes:
        product_of_primes *= p

    result = []
    # Iterate from 1 up to (product_of_primes - 1)
    for num in range(1, product_of_primes):
        is_relatively_prime = True
        for p in primes:
            if num % p == 0:
                is_relatively_prime = False
                break  # Not relatively prime to this prime, so not to the set
        if is_relatively_prime:
            result.append(num)

    return result

# Example usage:
# primes1 = [2, 3]
# print(f"Primes: {primes1}, Relatively prime numbers: {find_relatively_prime(primes1)}")

# primes2 = [3, 5]
# print(f"Primes: {primes2}, Relatively prime numbers: {find_relatively_prime(primes2)}")

# primes3 = [7]
# print(f"Primes: {primes3}, Relatively prime numbers: {find_relatively_prime(primes3)}")


Reminder to self:
- we skip 2, because otherwise $a^2$ and $b^2$ are both odd, and we can only factor even numbers.
- we skip 3, because if $a^2 \cong 1 \mod 3$, we only know that $a \cong \pm 1 \mod 3$, and it doesn't narrow anything down. (Similarly for $b^2$.)

In [90]:
# Unsurprisingly, using more primes is much slower...
primes = [5,7,11,13]
n = math.prod(primes)
x = find_relatively_prime(primes)


In [91]:
# get products of larger primes
primes2 = sympy.primerange(max(primes)+1, n-1)
nums2 = [[x1 * x2 for x1 in primes2] for x2 in primes2]
nums2 = [x1 for x2 in nums2 for x1 in x2 if x1 < n]
nums2 = set(nums2)

We loop through the possible values of $a$ and $b$.

In [92]:
matches = []
for a in tqdm(x):
  for b in x:
    diff = (a**2 - b**2) % n
    if diff in nums2:
      matches.append([diff, a, b, (a+b)%n, math.gcd(diff, a+b)])
matches = pd.DataFrame(matches, columns=["difference", "a", "b", "a+b", "GCD"])

100%|██████████| 2880/2880 [00:02<00:00, 1103.95it/s]


In [93]:
matches[:3]

,difference,a,b,a+b,GCD
0,3383,2,129,131,1
1,3383,2,311,313,1
2,3383,2,844,846,1


In [94]:
matches["a**2"] = (matches["a"] ** 2) % n
matches["b**2"] = (matches["b"] ** 2) % n
matches["divides"] = matches["difference"] % matches["a+b"] == 0
matches["has GCD"] = (matches["GCD"] != 1) & (matches["GCD"] != matches["difference"])
matches[:3]

,difference,a,b,a+b,GCD,a**2,b**2,divides,has GCD
0,3383,2,129,131,1,4,1626,False,False
1,3383,2,311,313,1,4,1626,False,False
2,3383,2,844,846,1,4,1626,False,False


The question now is: for a given $(a^2, b^2)$ pair, how often do the underlying $(a,b)$ yield a factor? (Occasionally, $a+b|x$; more often, we might hope that $\mathrm{GCD}(x, a+b)$ is nontrivial.)

Note that, given $(a^2, b^2)$, we still need to guess the sign of $\pm a$ and $\pm b$; so WLOG we don't separately consider $a-b$.

In [95]:
gcd_count = matches.groupby(["difference", "a**2", "b**2"]).agg({"has GCD": [len, "sum"]})
gcd_count = gcd_count.reset_index().sort_values(["difference", "a**2", "b**2"])
gcd_count

difference  a**2  b**2 has GCD    
                               len sum
0          323   324     1     256  22
1          323   779   456     256  30
2          323  1479  1156     256  27
3          323  1934  1611     256  25
4          323  4174  3851     256  26
..         ...   ...   ...     ...  ..
131       4777   991  1219     256  15
132       4777  1621  1849     256  16
133       4777  2531  2759     256  16
134       4777  3546  3774     256  16
135       4777  4456  4684     256  17

[136 rows x 5 columns]

To check this, we see if, having found $x = a^2 - b^2$, there is always some choice of $a$ and $b$ which yield a factor.

In [96]:
print((gcd_count["has GCD"] == 0).sum())


len    0
sum    0
dtype: int64


So, for these tiny numbers, this is the case. The **caveat**, of course, is that this is working mod a *tiny* set of primes...

Indeed, there seems to often be more than one choice of $a$ and $b$ which gives a nontrivial GCD. I don't see an obvious pattern to how many, though...

In [97]:
print(gcd_count["has GCD"].mean().round(3))

len    256.000
sum     17.515
dtype: float64
